# Parse Video Data

In [ ]:
import os
import numpy as np
import glob
import json
import cv2
from tqdm import tqdm
from torchvision import transforms
import torch
from torch.hub import load as hub_load
from sklearn.metrics.pairwise import cosine_similarity
from torchvision.io import read_image
import matplotlib.pyplot as plt
from PIL import Image

In [2]:
videos_dir = "/kaggle/input/news-event-retrieval-video-data"
all_video_paths = dict()
for folder_name in sorted(os.listdir(videos_dir)):
    # if part != "Videos_L21_a": continue
    data_part = folder_name.replace("Videos_", "")  # L21_a for ex
    all_video_paths[data_part] = dict()

for data_part in sorted(all_video_paths.keys()):
    data_part_path = f"{videos_dir}/Videos_{data_part}/video"
    video_paths = sorted(os.listdir(data_part_path))
    video_ids = [
        video_path.replace(".mp4", "").split("_")[-1] for video_path in video_paths
    ]
    for video_id, video_path in zip(video_ids, video_paths):
        video_path_full = f"{data_part_path}/{video_path}"
        all_video_paths[data_part][video_id] = video_path_full

In [7]:
def sample_frame_from_shot(start_idx, end_idx):
    s = start_idx
    e = end_idx
    frame_idxs = [
        int(s),
        int(s + (e - s) * 0.25),
        int(s + (e - s) * 0.5),
        int(s + (e - s) * 0.75),
        int(s + (e - s) * 1),
    ]
    return frame_idxs

# Extract frames

In [ ]:
scene_json_dirs = "/kaggle/input/scenes-segment"
save_dir_all = "./Keyframes"
if not os.path.exists(save_dir_all):
    os.mkdir(save_dir_all)

for key in all_video_paths.keys():
    save_dir = f"{save_dir_all}/{key}_extract"
    if key != "L21_a":
        continue
    if not os.path.exists(save_dir):
        os.mkdir(save_dir)

    video_paths_dict = all_video_paths[key]
    video_ids = sorted(video_paths_dict.keys())
    for video_id in tqdm(video_ids):
        video_path = video_paths_dict[video_id]
        video_scene_path = f"{scene_json_dirs}/{key}/{video_id}.json"

        with open(video_scene_path, "r") as f:
            video_scenes = json.load(f)

        if not os.path.exists(f"{save_dir}/{video_id}"):
            os.mkdir(f"{save_dir}/{video_id}")

        cap = cv2.VideoCapture(video_path)
        for i, shot in enumerate(tqdm(video_scenes)):
            shot_frames_id = sample_frame_from_shot(shot[0], shot[1])
            for index in shot_frames_id:
                cap.set(cv2.CAP_PROP_POS_FRAMES, index)
                filename = "{}/{:0>6d}.jpg".format(f"{save_dir}/{video_id}", index)
                ret, frame = cap.read()
                if ret:
                    if not cv2.imwrite(filename, frame):
                        print("fail save")
                else:
                    pass
        cap.release()

# Images Deduplication with DinoV2

In [ ]:
def show_keyframes_grid(folder_path, n=30, thumb_size=(120, 90), cols=10):
    """
    Hiển thị n keyframe đầu tiên từ thư mục theo dạng lưới.

    Parameters:
        folder_path (str): Đường dẫn tới thư mục chứa keyframes.
        n (int): Số frame hiển thị.
        thumb_size (tuple): Kích thước ảnh nhỏ (width, height).
        cols (int): Số cột trong lưới hiển thị.
    """
    # Lấy danh sách ảnh và sắp xếp
    image_files = sorted(
        [f for f in os.listdir(folder_path) if f.lower().endswith((".jpg", ".png"))]
    )[:n]

    # Tính số hàng cần
    rows = (len(image_files) + cols - 1) // cols

    # Thiết lập plot
    plt.figure(figsize=(cols * 1.5, rows * 1.5))

    for i, img_name in enumerate(image_files):
        img_path = os.path.join(folder_path, img_name)
        img = Image.open(img_path).resize(thumb_size)

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"{i}", fontsize=8)

    plt.tight_layout()
    plt.show()

In [ ]:
# Load DINOv2 từ torchvision
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14").to(device)
model.eval()
# Transform ảnh
transform = transforms.Compose(
    [
        transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

In [ ]:
def extract_features(image_paths):
    features = []
    for path in image_paths:
        try:
            # Read image as tensor and convert to PIL Image
            img = read_image(path)
            img = transforms.ToPILImage()(img)  # Convert tensor to PIL Image
            img = transform(img).unsqueeze(0).to(device)  # [1, 3, 224, 224]
            with torch.no_grad():
                feat = model(img)
                features.append(feat.squeeze().cpu().numpy())
        except Exception as e:
            print(f"Error reading {path}: {e}")
    return np.array(features)


def deduplicate_frames_in_folder(folder_path, threshold=0.9):
    image_files = sorted(
        [
            os.path.join(folder_path, f)
            for f in os.listdir(folder_path)
            if f.lower().endswith((".jpg", ".png"))
        ]
    )

    if len(image_files) <= 1:
        return

    print(f"🧠 Extracting features from {len(image_files)} frames in: {folder_path}")
    features = extract_features(image_files)

    # Tính cosine similarity
    print("🔍 Computing cosine similarity...")
    similarity_matrix = cosine_similarity(features)

    to_delete = set()
    for i in range(len(image_files)):
        for j in range(i + 1, len(image_files)):
            if similarity_matrix[i][j] >= threshold:
                # Nếu frame j giống frame i thì đánh dấu xóa j
                to_delete.add(image_files[j])

    # Xoá ảnh trùng lặp
    for f in to_delete:
        os.remove(f)
        print(f"🗑️ Removed duplicate frame: {os.path.basename(f)}")


def run_pipeline_on_all_videos(parent_folder, threshold=0.9):
    video_folders = sorted(
        [
            os.path.join(parent_folder, d)
            for d in os.listdir(parent_folder)
            if os.path.isdir(os.path.join(parent_folder, d))
        ]
    )

    for folder in tqdm(video_folders, desc="🎞️ Processing videos"):
        deduplicate_frames_in_folder(folder, threshold=threshold)

In [ ]:
run_pipeline_on_all_videos("/kaggle/working/Keyframes/L22_a_extract", threshold=0.8)